In [1]:
!pip install -U langchain langchain-community langchain-core langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.18
    Uninstalling langchain-core-1.2.18:
      Successfully uninstalled langchain-core-1.2.18
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google

In [2]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

In [3]:
# ----------------------------
# Configure Gemini API
# ----------------------------

# os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
# genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.

In [ ]:
# ---------------------------
# LLM
# ---------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


# ---------------------------
# Tool
# ---------------------------

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))


tools = [calculator]


# ---------------------------
# Prompt
# ---------------------------

prompt = ChatPromptTemplate.from_messages(
[
("system", "You are a helpful assistant. Use the calculator tool when needed."),
("placeholder", "{chat_history}"),
("human", "{input}")
]
)


# ---------------------------
# Tool binding
# ---------------------------

llm_with_tools = llm.bind_tools(tools)

In [ ]:



# ---------------------------
# Chain
# ---------------------------

chain = prompt | llm_with_tools


# ---------------------------
# Memory
# ---------------------------

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


agent = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)



In [ ]:

# ---------------------------
# Chat loop
# ---------------------------

while True:

    query = input("\nYou: ")

    if query.lower() == "exit":
        break

    response = agent.invoke(
        {"input": query},
        config={"configurable": {"session_id": "demo"}}
    )

    print("\nAgent:", response.content)